In [ ]:
%%sql -r dataframe_1
-- =====================================================
-- PHASE 4A: REDEMPTION STAGING TABLE
-- NOTEBOOK: NB_04_REDEMPTION_PROCESSING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA STG;

CREATE TABLE IF NOT EXISTS STG_REDEMPTION_TRANSACTIONS (
    -- Feed-level attributes
    MEMBER_ID               VARCHAR(18),
    FEED_DATE               DATE,

    -- Transaction attributes
    TXN_ID                  VARCHAR(100),
    TXN_DATE                DATE,
    PARTNER                 VARCHAR(255),
    MILES_REDEEMED          NUMBER(18,0),
    STATUS                  VARCHAR(50),

    -- Source metadata
    SOURCE_FILE_NAME        VARCHAR(500),
    FILE_TIMESTAMP          TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP     TIMESTAMP_NTZ,
    BATCH_ID                VARCHAR(100),
    SOURCE_RECORD_NUMBER    NUMBER(38,0),

    -- Data quality fields
    VALIDATION_STATUS       VARCHAR(20),
    REJECTION_REASON        VARCHAR(2000),
    PROCESSED_TIMESTAMP     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

DESCRIBE TABLE STG_REDEMPTION_TRANSACTIONS;

SHOW TABLES IN SCHEMA STG;

In [ ]:
%%sql -r dataframe_2
-- =====================================================
-- PHASE 4B: CORE REDEMPTION TRANSACTION TABLE
-- NOTEBOOK: NB_04_REDEMPTION_PROCESSING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA CORE;

CREATE TABLE IF NOT EXISTS CORE.REDEMPTION_TRANSACTIONS (
    MEMBER_ID               VARCHAR(18) NOT NULL,
    TXN_ID                  VARCHAR(100) NOT NULL,
    FEED_DATE               DATE,
    TXN_DATE                DATE,
    PARTNER                 VARCHAR(255),
    MILES_REDEEMED          NUMBER(18,0),
    STATUS                  VARCHAR(50),

    SOURCE_FILE_NAME        VARCHAR(500),
    FILE_TIMESTAMP          TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP     TIMESTAMP_NTZ,
    BATCH_ID                VARCHAR(100),
    SOURCE_RECORD_NUMBER    NUMBER(38,0),

    RECORD_CREATED_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RECORD_UPDATED_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

DESCRIBE TABLE CORE.REDEMPTION_TRANSACTIONS;

SHOW TABLES IN SCHEMA CORE;

In [ ]:
-- CELL 3: FLATTEN JSON REDEMPTION RECORDS INTO STAGING

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

INSERT INTO STG.STG_REDEMPTION_TRANSACTIONS (
    MEMBER_ID,
    FEED_DATE,
    TXN_ID,
    TXN_DATE,
    PARTNER,
    MILES_REDEEMED,
    STATUS,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    INGESTION_TIMESTAMP,
    BATCH_ID,
    SOURCE_RECORD_NUMBER,
    VALIDATION_STATUS,
    REJECTION_REASON
)
WITH FLATTENED AS (
    SELECT
        r.RAW_RECORD:member_id::VARCHAR(18) AS MEMBER_ID,

        TRY_TO_DATE(
            r.RAW_RECORD:feed_date::VARCHAR,
            'YYYYMMDD'
        ) AS FEED_DATE,

        f.VALUE:txn_id::VARCHAR(100) AS TXN_ID,

        TRY_TO_DATE(
            f.VALUE:txn_date::VARCHAR,
            'YYYYMMDD'
        ) AS TXN_DATE,

        f.VALUE:partner::VARCHAR(255) AS PARTNER,

        TRY_TO_NUMBER(
            f.VALUE:miles_redeemed::VARCHAR
        ) AS MILES_REDEEMED,

        f.VALUE:status::VARCHAR(50) AS STATUS,

        r.SOURCE_FILE_NAME,
        r.FILE_TIMESTAMP,
        r.INGESTION_TIMESTAMP,
        r.BATCH_ID,
        r.SOURCE_RECORD_NUMBER

    FROM RAW.RAW_REDEMPTIONS r,
         LATERAL FLATTEN(
             INPUT => r.RAW_RECORD:redemptions
         ) f

    WHERE r.SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%'
)
SELECT
    p.MEMBER_ID,
    p.FEED_DATE,
    p.TXN_ID,
    p.TXN_DATE,
    p.PARTNER,
    p.MILES_REDEEMED,
    p.STATUS,
    p.SOURCE_FILE_NAME,
    p.FILE_TIMESTAMP,
    p.INGESTION_TIMESTAMP,
    p.BATCH_ID,
    p.SOURCE_RECORD_NUMBER,

    CASE
        WHEN p.MEMBER_ID IS NULL OR TRIM(p.MEMBER_ID) = ''
            THEN 'REJECTED'
        WHEN p.TXN_ID IS NULL OR TRIM(p.TXN_ID) = ''
            THEN 'REJECTED'
        WHEN p.TXN_DATE IS NULL
            THEN 'REJECTED'
        WHEN p.MILES_REDEEMED IS NULL OR p.MILES_REDEEMED < 0
            THEN 'REJECTED'
        ELSE 'VALID'
    END,

    CASE
        WHEN p.MEMBER_ID IS NULL OR TRIM(p.MEMBER_ID) = ''
            THEN 'MEMBER_ID_MISSING'
        WHEN p.TXN_ID IS NULL OR TRIM(p.TXN_ID) = ''
            THEN 'TXN_ID_MISSING'
        WHEN p.TXN_DATE IS NULL
            THEN 'INVALID_TXN_DATE'
        WHEN p.MILES_REDEEMED IS NULL OR p.MILES_REDEEMED < 0
            THEN 'INVALID_MILES_REDEEMED'
        ELSE NULL
    END

FROM FLATTENED p
WHERE NOT EXISTS (
    SELECT 1
    FROM STG.STG_REDEMPTION_TRANSACTIONS s
    WHERE s.SOURCE_FILE_NAME = p.SOURCE_FILE_NAME
      AND s.MEMBER_ID = p.MEMBER_ID
      AND s.TXN_ID = p.TXN_ID
);

In [ ]:
%%sql -r dataframe_4
-- CELL 5: REDEMPTION QUALITY CHECKS

USE DATABASE SKYPOINTS_INCU;

-- 1. Check duplicate transaction keys
SELECT
    MEMBER_ID,
    TXN_ID,
    COUNT(*) AS RECORD_COUNT
FROM STG.STG_REDEMPTION_TRANSACTIONS
WHERE SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%'
GROUP BY MEMBER_ID, TXN_ID
HAVING COUNT(*) > 1;

-- 2. Check whether redemption members exist in CORE
SELECT
    R.MEMBER_ID,
    R.TXN_ID,
    R.MILES_REDEEMED
FROM STG.STG_REDEMPTION_TRANSACTIONS R
LEFT JOIN CORE.MEMBER_PROFILE M
    ON R.MEMBER_ID = M.MEMBER_ID
WHERE R.SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%'
  AND M.MEMBER_ID IS NULL;

-- 3. Summary of member matching
SELECT
    COUNT(*) AS TOTAL_TRANSACTIONS,
    COUNT_IF(M.MEMBER_ID IS NOT NULL) AS MATCHED_TRANSACTIONS,
    COUNT_IF(M.MEMBER_ID IS NULL) AS UNMATCHED_TRANSACTIONS
FROM STG.STG_REDEMPTION_TRANSACTIONS R
LEFT JOIN CORE.MEMBER_PROFILE M
    ON R.MEMBER_ID = M.MEMBER_ID
WHERE R.SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%';